# Sensitivity to Row Choice

## Imports

In [ ]:
import os
import sys
import time
import pickle
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append('../src')
from zoo import TASKS, FAMILY, META, SEED, reg_models, clf_models, make_pipeline, grid_prefix, load_task, make_folds

warnings.filterwarnings('ignore')
sns.set_theme(style='ticks', context='notebook')

import ast
import json
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, f1_score

## Settings

In [ ]:
TASKS_TO_RUN = list(TASKS)
ONLY_MODELS = None            # for example ['Ridge', 'RF'] for a quick test
zoo = pd.concat([pd.read_csv(f'../results/{t}__zoo_results.csv') for t in TASKS], ignore_index=True)

## Row Subsets

In [ ]:
def row_subsets(d):
    subsets = {'All rows': np.ones(len(d), bool),
               'Mean+Single only': d['Value_Type'].isin(['Mean', 'Single']).values}
    if d['Target_Is_Derived'].sum() > 0:
        subsets['Measured only'] = (d['Target_Is_Derived'] == 0).values
        subsets['Measured, Mean+Single'] = ((d['Target_Is_Derived'] == 0) & d['Value_Type'].isin(['Mean', 'Single'])).values
    return subsets

## Run

In [ ]:
def tuned_params(row, grid):
    # recover the tuned values of the main experiment (no re-tuning here)
    best = json.loads(row['Best_params'])
    params = {}
    for k, v in best.items():
        for candidate in grid[k]:
            if str(candidate) == v:
                params[k] = candidate
                break
        else:
            params[k] = ast.literal_eval(v)
    return params


records = []
for task in TASKS_TO_RUN:
    d0, _, _, _, is_clf, target = load_task(task)
    models = clf_models() if is_clf else reg_models()
    if ONLY_MODELS:
        models = {k: v for k, v in models.items() if k in ONLY_MODELS}
    for subset_name, mask in row_subsets(d0).items():
        d = d0[mask].reset_index(drop=True)
        X = d.drop(columns=META + [target])
        y = d[target].values
        g = d['Paper_ID'].values
        idx = np.arange(len(d))
        train, test = train_test_split(idx, test_size=0.2, random_state=SEED)
        folds = make_folds(g, 5, 7)
        for name, (model, grid, impute, scale_target) in models.items():
            row = zoo[(zoo['File'] == task) & (zoo['Model'] == name)].iloc[0]
            params = tuned_params(row, grid)
            build = lambda: make_pipeline(clone(model).set_params(**params), impute, scale_target, is_clf)
            try:
                p = build().fit(X.iloc[train], y[train]).predict(X.iloc[test])
                pooled = np.full(len(d), np.nan)
                for k in range(5):
                    a, b = np.where(folds != k)[0], np.where(folds == k)[0]
                    pooled[b] = build().fit(X.iloc[a], y[a]).predict(X.iloc[b])
            except Exception as e:
                print('error', task, subset_name, name, e)
                continue
            if is_clf:
                random_score = f1_score(y[test], p, average='macro')
                paperwise = f1_score(y, pooled.astype(int), average='macro')
                le25 = np.nan
            else:
                random_score, paperwise = r2_score(y[test], p), r2_score(y, pooled)
                small = y <= 25
                le25 = r2_score(y[small], pooled[small]) if task.endswith('MaxLoad') else np.nan
            records.append(dict(File=task, Subset=subset_name, Rows=len(d), Papers=len(set(g)), Model=name,
                                Random=random_score, Paperwise=paperwise, Paperwise_le25=le25))
        print(task, subset_name, len(d), 'done')
sensitivity = pd.DataFrame(records)
sensitivity.to_csv('../results/sensitivity.csv', index=False)

## Summary

In [ ]:
sensitivity.groupby(['File', 'Subset'])[['Random', 'Paperwise']].max().round(3)